# Chinese Rap NER — Visualization Dashboard

从 Google Drive `rap-data/` 文件夹读取 pipeline 产出的数据，生成全套可视化图表。

**前置条件**: 先跑完 `colab.ipynb`（至少到聚类那步），确保 Drive 中 `rap-data/` 目录有以下文件:
- `entities_long.csv`
- `artist_clusters.csv`
- `cluster_entity_summary.csv`
- `bag_of_entities.npz` + `.artists.csv` + `.entities.csv`

> 也支持本地运行：如果本地 `outputs/` 目录已有数据则直接读取，跳过 Drive 挂载。

In [ ]:
# === Step 0: 环境准备 ===
# Colab 依赖安装（本地有就跳过）
import subprocess, importlib
_VIZ_DEPS = ["matplotlib", "seaborn", "sklearn", "scipy", "networkx"]
for _dep in _VIZ_DEPS:
    try:
        importlib.import_module(_dep)
    except ImportError:
        subprocess.check_call(["pip", "install", "-q", _dep.replace("sklearn", "scikit-learn")])

# 可选依赖
for _opt in ["wordcloud", "adjustText"]:
    try:
        importlib.import_module(_opt)
    except ImportError:
        subprocess.check_call(["pip", "install", "-q", _opt])

In [ ]:
import shutil
from pathlib import Path

# ---- 数据路径：优先本地 outputs/，否则从 Drive 拷贝 ----
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)

_REQUIRED_FILES = [
    "entities_long.csv",
    "artist_clusters.csv",
    "cluster_entity_summary.csv",
    "bag_of_entities.npz",
    "bag_of_entities.artists.csv",
    "bag_of_entities.entities.csv",
]

_local_ready = all((OUTPUTS / f).exists() for f in _REQUIRED_FILES)

if not _local_ready:
    # 在 Colab 上从 Drive 拷贝
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA = Path("/content/drive/MyDrive/rap-data")

    print("Copying data from Drive → outputs/ ...")
    for f in _REQUIRED_FILES:
        src = DRIVE_DATA / f
        if src.exists():
            shutil.copy(src, OUTPUTS / f)
            print(f"  ✓ {f}")
        else:
            print(f"  ✗ {f} NOT FOUND in Drive!")
    print("Done.")
else:
    print(f"[OK] All data found in local {OUTPUTS}/")

In [ ]:
import glob as _glob
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz

# ---- 中文字体配置 (Colab 兼容) ----
import subprocess as _sp
from matplotlib.font_manager import fontManager, FontProperties

# 候选字体名
_CN_FONTS = [
    "Noto Sans CJK SC", "Noto Sans SC",
    "WenQuanYi Micro Hei", "WenQuanYi Zen Hei",
    "SimHei", "Microsoft YaHei",
    "PingFang SC", "Heiti SC",
    "AR PL UMing CN", "AR PL UKai CN",
]

_available = {f.name for f in fontManager.ttflist}
_CN_FONT = next((f for f in _CN_FONTS if f in _available), None)

if _CN_FONT is None:
    # Colab / Debian: 安装 Noto CJK 字体
    _sp.run(["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"], capture_output=True)
    # 搜索所有可能的安装位置
    for _fp in _glob.glob("/usr/share/fonts/**/Noto*CJK*.ttc", recursive=True):
        fontManager.addfont(_fp)
    for _fp in _glob.glob("/usr/share/fonts/**/Noto*CJK*.otf", recursive=True):
        fontManager.addfont(_fp)
    # 重新检查
    _available = {f.name for f in fontManager.ttflist}
    _CN_FONT = next((f for f in _CN_FONTS if f in _available), "DejaVu Sans")

# 清除 matplotlib 字体缓存，确保新字体被识别
matplotlib.font_manager._load_fontmanager(try_read_cache=False)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [_CN_FONT, "DejaVu Sans"],
    "axes.unicode_minus": False,
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
})
print(f"Using font: {_CN_FONT}")

FIGS = Path("figs")
FIGS.mkdir(exist_ok=True)

# === 加载数据 ===
entity_df = pd.read_csv(OUTPUTS / "entities_long.csv")
clusters  = pd.read_csv(OUTPUTS / "artist_clusters.csv")
summary   = pd.read_csv(OUTPUTS / "cluster_entity_summary.csv")

boe_data    = load_npz(str(OUTPUTS / "bag_of_entities.npz"))
boe_artists = pd.read_csv(OUTPUTS / "bag_of_entities.artists.csv")["artist"].tolist()
boe_entities = pd.read_csv(OUTPUTS / "bag_of_entities.entities.csv")["entity"].tolist()

# 过滤掉 "keep" 标签（如果 entity_review.csv 中 keep 被误当作标签）
if "keep" in entity_df["label"].values:
    print(f"[WARN] Found 'keep' label ({(entity_df['label']=='keep').sum()} rows), restoring original labels...")
    # keep 标签的实体 -> 查找它们在其他行中的真实标签
    keep_mask = entity_df["label"] == "keep"
    keep_entities = entity_df.loc[keep_mask, "entity"].unique()
    # 对于每个 keep 实体，找它在非 keep 行中最常见的标签
    non_keep = entity_df[~keep_mask]
    for ent in keep_entities:
        real_labels = non_keep[non_keep["entity"] == ent]["label"]
        if len(real_labels) > 0:
            real_label = real_labels.mode().iloc[0]
            entity_df.loc[keep_mask & (entity_df["entity"] == ent), "label"] = real_label
    # 如果还有剩余的 keep（没有其他行参考），标记为 MISC
    entity_df.loc[entity_df["label"] == "keep", "label"] = "MISC"
    print(f"  Fixed. Labels now: {entity_df['label'].nunique()} types")

print(f"Entities: {len(entity_df)} mentions, {entity_df['entity'].nunique()} unique")
print(f"Artists:  {clusters['artist'].nunique()}")
print(f"Clusters: {clusters['cluster'].nunique()}")
print(f"BOE matrix: {boe_data.shape}")

In [ ]:
label_counts = entity_df["label"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- 左: 柱状图 (所有标签) ---
colors = sns.color_palette("Set2", len(label_counts))
axes[0].barh(label_counts.index[::-1], label_counts.values[::-1], color=colors[::-1])
axes[0].set_xlabel("Mention count")
axes[0].set_title("Entity Label Distribution (mentions)")
for i, v in enumerate(label_counts.values[::-1]):
    axes[0].text(v + 20, i, str(v), va="center", fontsize=8)

# --- 右: 饼图 (合并小类别) ---
# 占比 < 2% 的合并为 OTHER
total = label_counts.sum()
threshold = total * 0.02
major = label_counts[label_counts >= threshold]
minor_sum = label_counts[label_counts < threshold].sum()
if minor_sum > 0:
    pie_data = pd.concat([major, pd.Series({"OTHER": minor_sum})])
else:
    pie_data = major

pie_colors = sns.color_palette("Set2", len(pie_data))
wedges, texts, autotexts = axes[1].pie(
    pie_data.values, labels=pie_data.index, autopct="%1.1f%%",
    colors=pie_colors, startangle=140, pctdistance=0.8,
    textprops={"fontsize": 9},
)
for t in autotexts:
    t.set_fontsize(8)
axes[1].set_title("Entity Label Proportions")

plt.tight_layout()
fig.savefig(FIGS / "label_distribution.png")
plt.show()
print(f"({len(label_counts) - len(major)} labels with <2% merged into OTHER)")

## 2. Top-30 高频实体

In [ ]:
top_n = 30
top_entities = (
    entity_df.groupby(["entity", "label"]).size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(top_n)
)

# 颜色映射: label → color
all_labels = entity_df["label"].unique()
label_cmap = dict(zip(all_labels, sns.color_palette("Set2", len(all_labels))))
bar_colors = [label_cmap[l] for l in top_entities["label"]]

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = range(len(top_entities))
ax.barh(y_pos, top_entities["count"].values, color=bar_colors)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{r['entity']} [{r['label']}]" for _, r in top_entities.iterrows()],
                   fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Frequency")
ax.set_title(f"Top {top_n} 高频实体")

# 图例
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=label_cmap[l], label=l) for l in all_labels if l in top_entities["label"].values]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9)

plt.tight_layout()
fig.savefig(FIGS / "top_entities.png")
plt.show()

## 3. 各 Cluster 代表实体柱状图

In [ ]:
cluster_names = sorted(summary["cluster"].unique())
n_clusters = len(cluster_names)

# 每个 cluster 展示 top 10 实体
top_k = 10
n_cols = 3
n_rows = (n_clusters + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

cluster_colors = sns.color_palette("tab10", n_clusters)

for i, cname in enumerate(cluster_names):
    ax = axes[i]
    cdata = summary[summary["cluster"] == cname].head(top_k)
    cdata = cdata[cdata["centroid_weight"] > 0]
    n_artists = (clusters["cluster"] == int(cname.split("_")[1])).sum()

    ax.barh(range(len(cdata)), cdata["centroid_weight"].values,
            color=cluster_colors[i], alpha=0.85)
    ax.set_yticks(range(len(cdata)))
    ax.set_yticklabels(cdata["entity"].values, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(f"{cname} ({n_artists} artists)", fontsize=11, fontweight="bold")
    ax.set_xlabel("centroid weight", fontsize=8)

# 关闭多余的子图
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("各 Cluster 代表实体 (Top 10)", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(FIGS / "cluster_top_entities.png")
plt.show()

## 4. 各 Cluster 的标签组成 (堆叠柱状图)

In [ ]:
# 把 entity_df 和 cluster assignments 合并，统计每个 cluster 的标签分布
entity_with_cluster = entity_df.merge(clusters, on="artist", how="inner")

cross = pd.crosstab(entity_with_cluster["cluster"], entity_with_cluster["label"])
cross_pct = cross.div(cross.sum(axis=1), axis=0)

# 标签太多时只保留 top N，其余合并为 OTHER
TOP_LABELS = 8
if cross.shape[1] > TOP_LABELS:
    top_labels = cross.sum().nlargest(TOP_LABELS).index.tolist()
    other_cols = [c for c in cross.columns if c not in top_labels]
    cross_grouped = cross[top_labels].copy()
    cross_grouped["OTHER"] = cross[other_cols].sum(axis=1)
    cross_pct_grouped = cross_grouped.div(cross_grouped.sum(axis=1), axis=0)
else:
    cross_grouped = cross
    cross_pct_grouped = cross_pct

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cpal = sns.color_palette("Set2", cross_grouped.shape[1])

cross_grouped.plot(kind="bar", stacked=True, ax=axes[0], color=cpal)
axes[0].set_title("Cluster Label Counts")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Mentions")
axes[0].legend(fontsize=7, loc="upper right", ncol=2)
axes[0].tick_params(axis='x', rotation=0)

cross_pct_grouped.plot(kind="bar", stacked=True, ax=axes[1], color=cpal)
axes[1].set_title("Cluster Label Proportions")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Proportion")
axes[1].legend(fontsize=7, loc="upper right", ncol=2)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
fig.savefig(FIGS / "cluster_label_composition.png")
plt.show()

## 5. t-SNE 降维散点图 — 艺人聚类分布

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

# L2 归一化 (与聚类时一致)
mat_norm = normalize(boe_data, norm="l2")

# t-SNE 降到 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(boe_artists) - 1),
            init="pca", learning_rate="auto")
coords = tsne.fit_transform(mat_norm.toarray())

tsne_df = pd.DataFrame({
    "artist": boe_artists,
    "x": coords[:, 0],
    "y": coords[:, 1],
})
tsne_df = tsne_df.merge(clusters, on="artist", how="left")
print(f"t-SNE done: {len(tsne_df)} artists")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

scatter_colors = sns.color_palette("tab10", n_clusters)
for i, cid in enumerate(sorted(tsne_df["cluster"].unique())):
    subset = tsne_df[tsne_df["cluster"] == cid]
    ax.scatter(subset["x"], subset["y"], c=[scatter_colors[i]], label=f"cluster_{cid}",
              s=60, alpha=0.75, edgecolors="white", linewidth=0.5)

# 标注艺人名（只标注离质心较远的，避免重叠）
texts = []
for _, row in tsne_df.iterrows():
    texts.append(ax.text(row["x"], row["y"], row["artist"], fontsize=6, alpha=0.7))

# 尝试用 adjustText 调整标签位置，如果安装了的话
try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3),
                max_move=(20, 20))
except ImportError:
    pass  # 没有 adjustText 也没关系，只是标签可能重叠

ax.legend(fontsize=9, loc="best")
ax.set_title("t-SNE: 中文 Rap 歌手聚类分布", fontsize=14)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")

plt.tight_layout()
fig.savefig(FIGS / "tsne_clusters.png")
plt.show()

## 6. Cluster 规模与实体丰富度

In [ ]:
# 每个 cluster 的 artist 数、平均实体数、总实体数
entity_per_artist = entity_with_cluster.groupby(["cluster", "artist"]).size().reset_index(name="n_mentions")
cluster_stats = entity_per_artist.groupby("cluster").agg(
    n_artists=("artist", "nunique"),
    avg_mentions=("n_mentions", "mean"),
    total_mentions=("n_mentions", "sum"),
).reset_index()

# 每个 cluster 的 unique entity 数
unique_ents = entity_with_cluster.groupby("cluster")["entity"].nunique().reset_index(name="unique_entities")
cluster_stats = cluster_stats.merge(unique_ents, on="cluster")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 艺人数
axes[0].bar(cluster_stats["cluster"], cluster_stats["n_artists"],
            color=scatter_colors)
axes[0].set_title("每个 Cluster 的艺人数")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Artists")

# 每人平均 mention 数
axes[1].bar(cluster_stats["cluster"], cluster_stats["avg_mentions"],
            color=scatter_colors)
axes[1].set_title("每人平均实体 mention 数")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Avg mentions/artist")

# unique 实体数
axes[2].bar(cluster_stats["cluster"], cluster_stats["unique_entities"],
            color=scatter_colors)
axes[2].set_title("每个 Cluster 的独立实体数")
axes[2].set_xlabel("Cluster")
axes[2].set_ylabel("Unique entities")

plt.tight_layout()
fig.savefig(FIGS / "cluster_stats.png")
plt.show()

print(cluster_stats.to_string(index=False))

## 7. 实体共现热力图 (Top 实体 × Cluster)

In [ ]:
# 取全局 top 25 实体，看它们在各 cluster 中的分布
top25_entities = entity_df["entity"].value_counts().head(25).index.tolist()

heat_data = (
    entity_with_cluster[entity_with_cluster["entity"].isin(top25_entities)]
    .groupby(["entity", "cluster"]).size()
    .reset_index(name="count")
    .pivot(index="entity", columns="cluster", values="count")
    .fillna(0)
)

# 按总频率排序
heat_data = heat_data.loc[heat_data.sum(axis=1).sort_values(ascending=False).index]
heat_data.columns = [f"cluster_{c}" for c in heat_data.columns]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(heat_data, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax,
            linewidths=0.5, cbar_kws={"label": "Mentions"})
ax.set_title("Top 25 实体 × Cluster 分布热力图", fontsize=13)
ax.set_ylabel("")

plt.tight_layout()
fig.savefig(FIGS / "entity_cluster_heatmap.png")
plt.show()

## 8. 词云 — 每个 Cluster

In [ ]:
try:
    from wordcloud import WordCloud
    _HAS_WORDCLOUD = True
except ImportError:
    print("wordcloud not installed, skipping. Install with: pip install wordcloud")
    _HAS_WORDCLOUD = False

if _HAS_WORDCLOUD:
    # 查找中文字体文件
    import glob
    _font_candidates = [
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/noto-cjk/NotoSansCJK-Regular.ttc",
        "/System/Library/Fonts/PingFang.ttc",
        "C:/Windows/Fonts/msyh.ttc",
    ]
    # Also search common locations
    _font_candidates += glob.glob("/usr/share/fonts/**/Noto*CJK*.ttc", recursive=True)
    _font_candidates += glob.glob("/usr/share/fonts/**/Noto*CJK*.otf", recursive=True)

    _wc_font = None
    for fp in _font_candidates:
        if Path(fp).exists():
            _wc_font = fp
            break

    if _wc_font is None:
        print("Warning: No CJK font found for word cloud. Chinese text may not render.")

    n_cols = 3
    n_rows = (n_clusters + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    axes = axes.flatten()

    for i, cid in enumerate(sorted(entity_with_cluster["cluster"].unique())):
        subset = entity_with_cluster[entity_with_cluster["cluster"] == cid]
        freq_dict = subset["entity"].value_counts().to_dict()

        wc_kwargs = dict(
            width=800, height=500,
            background_color="white",
            max_words=80,
            colormap="tab10",
            prefer_horizontal=0.7,
        )
        if _wc_font:
            wc_kwargs["font_path"] = _wc_font

        wc = WordCloud(**wc_kwargs).generate_from_frequencies(freq_dict)

        n_artists = (clusters["cluster"] == cid).sum()
        axes[i].imshow(wc, interpolation="bilinear")
        axes[i].set_title(f"Cluster {cid} ({n_artists} artists)", fontsize=12, fontweight="bold")
        axes[i].axis("off")

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("各 Cluster 实体词云", fontsize=15, y=1.01)
    plt.tight_layout()
    fig.savefig(FIGS / "cluster_wordclouds.png")
    plt.show()

## 9. 实体共现网络图

In [ ]:
from collections import Counter
from itertools import combinations

# 对每个 artist，取其 top 实体，构建共现边
TOP_PER_ARTIST = 8  # 每个 artist 取 top N 实体
MIN_EDGE_WEIGHT = 3  # 至少被 N 个 artist 共同提及

artist_top_entities = (
    entity_df.groupby(["artist", "entity"]).size()
    .reset_index(name="count")
    .sort_values(["artist", "count"], ascending=[True, False])
    .groupby("artist").head(TOP_PER_ARTIST)
)

# 统计共现
cooccur = Counter()
for artist, grp in artist_top_entities.groupby("artist"):
    ents = grp["entity"].tolist()
    for a, b in combinations(sorted(ents), 2):
        cooccur[(a, b)] += 1

# 过滤低权重边
edges = [(a, b, w) for (a, b), w in cooccur.items() if w >= MIN_EDGE_WEIGHT]
print(f"Co-occurrence edges (weight >= {MIN_EDGE_WEIGHT}): {len(edges)}")

# 取涉及的节点
nodes_in_edges = set()
for a, b, _ in edges:
    nodes_in_edges.add(a)
    nodes_in_edges.add(b)
print(f"Nodes in network: {len(nodes_in_edges)}")

In [ ]:
import networkx as nx

G = nx.Graph()
for a, b, w in edges:
    G.add_edge(a, b, weight=w)

# 节点大小 = 全局频率
global_freq = entity_df["entity"].value_counts()
node_sizes = [max(global_freq.get(n, 1) * 0.8, 20) for n in G.nodes()]

# 节点颜色 = 标签
entity_label_map = entity_df.drop_duplicates("entity").set_index("entity")["label"].to_dict()
node_colors = [label_cmap.get(entity_label_map.get(n, "OTHER"), "gray") for n in G.nodes()]

# 边宽度 = 权重
edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
max_w = max(edge_weights) if edge_weights else 1
edge_widths = [0.5 + 2.5 * (w / max_w) for w in edge_weights]

fig, ax = plt.subplots(figsize=(14, 12))
pos = nx.spring_layout(G, k=1.8, iterations=60, seed=42)

nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.3, width=edge_widths, edge_color="gray")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes, node_color=node_colors,
                       alpha=0.8, edgecolors="white", linewidths=0.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7,
                        font_family=_CN_FONT if _CN_FONT else "sans-serif")

# 图例
legend_handles = [Patch(facecolor=label_cmap[l], label=l)
                  for l in all_labels if l in set(entity_label_map.get(n) for n in G.nodes())]
ax.legend(handles=legend_handles, loc="upper left", fontsize=9)

ax.set_title("实体共现网络图", fontsize=14)
ax.axis("off")

plt.tight_layout()
fig.savefig(FIGS / "entity_cooccurrence_network.png")
plt.show()

## 10. 每个 Cluster 的标签雷达图

In [ ]:
# 归一化的 label 分布 per cluster (用 grouped 版本，避免太多维度)
radar_data = cross_pct_grouped.copy()
labels_list = radar_data.columns.tolist()
n_labels = len(labels_list)

angles = np.linspace(0, 2 * np.pi, n_labels, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, (cid, row) in enumerate(radar_data.iterrows()):
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=1.5, label=f"cluster_{cid}",
            color=scatter_colors[i % len(scatter_colors)], alpha=0.8)
    ax.fill(angles, values, alpha=0.1, color=scatter_colors[i % len(scatter_colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_list, fontsize=9)
ax.set_title("Cluster Label Radar", fontsize=14, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)

plt.tight_layout()
fig.savefig(FIGS / "cluster_radar.png")
plt.show()

## 11. 艺人实体丰富度分布

In [ ]:
artist_entity_stats = entity_df.groupby("artist").agg(
    total_mentions=("entity", "size"),
    unique_entities=("entity", "nunique"),
    unique_labels=("label", "nunique"),
).reset_index()
artist_entity_stats = artist_entity_stats.merge(clusters, on="artist", how="left")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: total mentions vs unique entities, colored by cluster
for i, cid in enumerate(sorted(artist_entity_stats["cluster"].dropna().unique())):
    sub = artist_entity_stats[artist_entity_stats["cluster"] == cid]
    axes[0].scatter(sub["total_mentions"], sub["unique_entities"],
                    c=[scatter_colors[int(cid)]], label=f"cluster_{int(cid)}",
                    s=40, alpha=0.7, edgecolors="white", linewidth=0.3)

axes[0].set_xlabel("Total entity mentions")
axes[0].set_ylabel("Unique entities")
axes[0].set_title("艺人实体丰富度 (mention vs unique)")
axes[0].legend(fontsize=8)

# Histogram: unique entities distribution
axes[1].hist(artist_entity_stats["unique_entities"], bins=30,
             color="steelblue", edgecolor="white", alpha=0.8)
axes[1].axvline(artist_entity_stats["unique_entities"].median(),
                color="red", linestyle="--", label=f"median={artist_entity_stats['unique_entities'].median():.0f}")
axes[1].set_xlabel("Unique entities per artist")
axes[1].set_ylabel("Count")
axes[1].set_title("独立实体数分布")
axes[1].legend()

plt.tight_layout()
fig.savefig(FIGS / "artist_entity_richness.png")
plt.show()

## Summary

所有图表已保存到 `figs/` 目录：

| 图表 | 文件 | 说明 |
|------|------|------|
| 标签分布 | `label_distribution.png` | 各实体类型的频率和占比 |
| Top 实体 | `top_entities.png` | 全局高频实体 Top 30 |
| Cluster 代表实体 | `cluster_top_entities.png` | 每个 cluster 的 top 10 实体 |
| Cluster 标签组成 | `cluster_label_composition.png` | 堆叠柱状图 |
| t-SNE 散点图 | `tsne_clusters.png` | 艺人聚类 2D 分布 |
| Cluster 统计 | `cluster_stats.png` | 规模、丰富度对比 |
| 热力图 | `entity_cluster_heatmap.png` | Top 实体在各 cluster 中的分布 |
| 词云 | `cluster_wordclouds.png` | 每个 cluster 的实体词云 |
| 共现网络 | `entity_cooccurrence_network.png` | 实体共现关系 |
| 雷达图 | `cluster_radar.png` | 各 cluster 标签风格对比 |
| 艺人丰富度 | `artist_entity_richness.png` | 实体丰富度散点图和分布 |

In [ ]:
# === 保存图表到 Drive (Colab) ===
import shutil
from pathlib import Path

DRIVE_FIGS = Path("/content/drive/MyDrive/rap-data/figs")
if Path("/content/drive/MyDrive").exists():
    DRIVE_FIGS.mkdir(parents=True, exist_ok=True)
    for fig_file in FIGS.glob("*.png"):
        shutil.copy(fig_file, DRIVE_FIGS / fig_file.name)
    print(f"Saved {len(list(FIGS.glob('*.png')))} figures to Drive: {DRIVE_FIGS}")
else:
    print(f"Not on Colab. Figures saved locally in {FIGS}/")